# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset, as described by a Croissant schema, using the [`mlcroissant`](https://mlcommons.org/croissant) library.

### Dataset Source
The dataset metadata is loaded from a Croissant schema URL, and all field and record set references are by their `@id` for reproducibility and clarity.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display basic info about the dataset
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n{metadata.description}")

## 2. Data Overview

Review the available record sets, their fields, and their `@id` identifiers.

In Croissant datasets, each `RecordSet` contains the main data tables and each has a unique `@id`. We'll list all record sets, their IDs, and the fields within.


In [ ]:
# List all record sets and fields by their @id
print("Available RecordSets and their Fields (by @id):\n")
record_sets = list(dataset.record_sets)

record_set_ids = []
for record_set in record_sets:
    rid = record_set.id  # The @id of the RecordSet
    record_set_ids.append(rid)
    print(f"- RecordSet @id: {rid}")
    if hasattr(record_set, 'fields') and record_set.fields:
        for f in record_set.fields:
            print(f"    - Field @id: {f.id} (name: {getattr(f, 'name', '')})")
    print()
# Preview a few records from each RecordSet
for rid in record_set_ids:
    print(f"First record from RecordSet {rid}:")
    for i, rec in enumerate(dataset.records(record_set=rid)):
        print(rec)
        if i >= 0:
            break
    print("----------")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. All field extraction uses the `@id` for referencing, as above.

**Note:** You may select a RecordSet's `@id` from the cell above for focused analysis.

In [ ]:
# Extract data from every major RecordSet into a Pandas DataFrame
dataframes = {}
for rid in record_set_ids:
    # Extract all records from this RecordSet
    records = list(dataset.records(record_set=rid))
    df = pd.DataFrame(records)
    dataframes[rid] = df
    print(f"Loaded {len(df)} records for RecordSet id: {rid}")
    print(f"Columns (@id): {list(df.columns)}\n")
# Choose the primary RecordSet for further exploration
main_record_set = record_set_ids[0] if record_set_ids else None
if main_record_set and main_record_set in dataframes:
    display(dataframes[main_record_set].head())

## 4. Exploratory Data Analysis (EDA)

Let's perform common EDA steps on the main table. We'll:
 - Filter records by a numeric field
 - Normalize a numeric column
 - Group by a categorical field (if present)

Please adjust the chosen field `@id`s as found above.

In [ ]:
# Please specify the RecordSet and numeric field @ids for EDA
# Here we auto-detect the first numeric-looking field for demo
import numpy as np

df = dataframes[main_record_set]
numeric_field_id = None
for col in df.columns:
    # Try to infer a numeric column
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Attempt to convert columns to numeric
    for col in df.columns:
        coerced = pd.to_numeric(df[col], errors='coerce')
        if coerced.notna().sum() > 0:
            df[col] = coerced
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
print(f"Numeric field selected for analysis: {numeric_field_id}")

# Simple threshold for filtering
if numeric_field_id:
    threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 1
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} (mean): {len(filtered_df)} records")
    if not filtered_df.empty:
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by the first non-numeric (categorical) column
    group_field = None
    for col in df.columns:
        if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col]):
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"\nMean {numeric_field_id} grouped by {group_field}:")
        print(grouped_df.head())

## 5. Visualization

Visualize the numeric field distribution and any relationships with a categorical field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Histogram of numeric field
if numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, color='skyblue', kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Boxplot grouped by categorical field, if available
if group_field and numeric_field_id:
    n_categories = df[group_field].nunique()
    if n_categories <= 20:  # Only plot if few categories
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we have:
1. Loaded the Croissant schema and data records from the FAIR^2 dataset (`mlcroissant`).
2. Explored the structure of the data, using only `@id` referencing for all sets and fields for clarity and reproducibility.
3. Loaded each RecordSet as a DataFrame and performed EDA, including filtering and normalization by field @id.
4. Generated basic visualizations of the numeric data.

You can build upon this notebook to perform further statistical analysis and model development, referencing all schema components by their `@id`. For more information about Croissant and FAIR^2, see the [dataset catalog](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and the [`mlcroissant`](https://github.com/mlcommons/croissant) documentation.